# Crossmatch performance by tool (2026 rerun on SLAC SDF)

Reads the per-run JSON files written by `run_benchmark.py` under `results/<config>/<step>/{lsdb_w<w>,astropy,smatch}_run<i>.json`
and reproduces the layout of the Feb 2025 figure (`../lsdb_crossmatch_benchmarking/plot_benchmark.ipynb`).

Plotted quantity per tool, per ladder step:
* **LSDB (w workers)** = wall time of `crossmatch(...).map_partitions(len).compute()` (reads + matches).
* **Astropy** = pyarrow read of the same ZTF / Gaia / Gaia-margin files + cone filter + `match_coordinates_sky`
  + distance cut. `SkyCoord` construction is excluded, as in 2025.
* **smatch** = the same pyarrow read + `smatch.match(gaia, r, ztf, maxmatch=0)` and a per-ZTF closest-pair reduction
  (Gaia indexed; the 2025 call form with ZTF indexed hits a slow path on spatially sorted input, see NOTES.md).

Point = median over repeats; error bars = 10th/90th percentile; x = number of crossmatched rows.


In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("results")
# (config, series, step) triples left out of the curves. Both were run while the 16 x 8 GB Dask workers
# hit their 95% memory budget and were restarted (40 restarts in total, run_all log 18:00-20:45 on 2026-09-15):
# tasks were recomputed inside the timed region and the two repeats at pixels:1791 disagree by 2x
# (852 s vs 1668 s). The JSONs stay under results/default/ and are listed in the "excluded" table below.
EXCLUDE = {("default", "lsdb_16", "pixels:1024"), ("default", "lsdb_16", "pixels:1791")}
CONFIG_ALL = "default"   # ZTF default columns + 27 Gaia columns
CONFIG_RADEC = "radec"   # only ra, dec on both sides
CONFIG_SLURM = "default_slurm"  # default columns, the largest steps re-run on a dedicated SLURM node (optional)
OUT_PNG = "crossmatching-performance-2026.png"


In [ ]:
def load_runs(config):
    """All per-(step, component, repeat) records of one config, plus its settings."""
    recs = [json.loads(f.read_text()) for f in sorted((RESULTS / config).glob("*/*_run*.json"))]
    settings = json.loads((RESULTS / config / "settings.json").read_text())
    return recs, settings


def series_from_recs(recs, config):
    """-> {series_key: {step: [(time, result_length), ...]}}; (config, series, step) in EXCLUDE are dropped"""
    out = defaultdict(lambda: defaultdict(list))
    for r in recs:
        step = r["step"]
        if r["tool"] == "lsdb":
            xm = r.get("lsdb_crossmatch")
            if xm and xm.get("result_length") is not None and (config, f"lsdb_{r['workers']}", step) not in EXCLUDE:
                out[f"lsdb_{r['workers']}"][step].append((xm["time"], xm["result_length"]))
        elif r.get("ok"):
            tool = r["tool"]; ms = r["measurements"]
            io = sum(v["time"] for k, v in ms.items() if isinstance(v, dict) and (k.endswith("_pyarrow") or k.endswith("_io")))
            io += ms.get("cone_filter", {}).get("time", 0.0)
            xm = ms[f"{tool}_xmatch"]
            out[f"{tool}+io"][step].append((io + xm["time"], xm["result_length"]))
            out[f"{tool}_xmatch_only"][step].append((xm["time"], xm["result_length"]))
            out["pyarrow_io"][step].append((io, xm["result_length"]))
    return out


def failures(recs):
    """Steps where astropy/smatch did not finish (out of memory or skipped)."""
    rows = [{"step": r["step"], "tool": r["tool"], "run": r["run"], "oom": r.get("oom"), "skipped": r.get("skipped"),
             "error": str(r.get("error", ""))[-120:]} for r in recs if r["tool"] != "lsdb" and not r.get("ok")]
    return pd.DataFrame(rows)


def aggregate(series):
    """-> {series_key: DataFrame[step, n, rows, median, q10, q90]} sorted by rows"""
    agg = {}
    for key, per_step in series.items():
        rows = []
        for step, vals in per_step.items():
            t = np.array([v[0] for v in vals]); n = np.array([v[1] for v in vals])
            rows.append({"step": step, "n": len(t), "rows": int(np.median(n)), "median": np.median(t),
                         "q10": np.quantile(t, 0.1), "q90": np.quantile(t, 0.9)})
        agg[key] = pd.DataFrame(rows).sort_values("rows").reset_index(drop=True)
    return agg


recs_all, settings_all = load_runs(CONFIG_ALL)
agg = aggregate(series_from_recs(recs_all, CONFIG_ALL))
settings_slurm = None
for cfg, suffix in ((CONFIG_RADEC, "radec"), (CONFIG_SLURM, "slurm")):
    try:
        recs_cfg, settings_cfg = load_runs(cfg)
    except FileNotFoundError:
        print(f"no {cfg} config")
        continue
    if suffix == "slurm":
        settings_slurm = settings_cfg
    for k, df in aggregate(series_from_recs(recs_cfg, cfg)).items():
        if k.startswith("lsdb_"):
            agg[f"{k}_{suffix}"] = df
print(sorted(agg))
failures(recs_all)


In [ ]:
# Runs excluded from the curves (see EXCLUDE above): kept on disk, shown here
excluded = pd.DataFrame(
    [{"config": CONFIG_ALL, "step": r["step"], "workers": r["workers"], "run": r["run"],
      "time_s": round(r["lsdb_crossmatch"]["time"], 1), "rows": r["lsdb_crossmatch"]["result_length"]}
     for r in recs_all if r["tool"] == "lsdb" and (CONFIG_ALL, f"lsdb_{r['workers']}", r["step"]) in EXCLUDE]
)
excluded


In [ ]:
# result_length must agree across tools for every step (a mismatch is a bug, not noise)
check = {}
for key, df in agg.items():
    if key.endswith("_xmatch_only") or key == "pyarrow_io":
        continue
    check[key] = df.set_index("step")["rows"]
pd.DataFrame(check)


In [ ]:
# Median-time table (seconds)
tbl = {k: v.set_index("rows")["median"].round(2) for k, v in agg.items() if not k.endswith("_xmatch_only")}
pd.DataFrame(tbl).sort_index()


In [ ]:
plt.style.use("default")
plt.rcParams.update({"font.size": 16, "axes.labelsize": 20, "axes.titlesize": 20, "legend.fontsize": 14})
fig, ax = plt.subplots(1, 1, figsize=(11, 7.5))

plotting_args = {
    "lsdb_1": dict(marker="o", label="LSDB (1 worker)", c="#012970", ls="-", lw=2),
    "lsdb_4": dict(marker="o", label="LSDB (4 workers)", c="#012970", ls="--", lw=2),
    "lsdb_16": dict(marker="o", label="LSDB (16 workers)", c="#012970", ls="-.", lw=2),
    "lsdb_16_radec": dict(marker="o", label="LSDB (16 workers, only RA and Dec columns)", c="#A74482", ls="-.", lw=2),
    "lsdb_16_slurm": dict(marker="s", label="LSDB (16 workers, dedicated SLURM node)", c="#012970", ls=":", lw=2, ms=9),
    "astropy+io": dict(marker="o", label="Astropy", c="orange", ls="-", lw=2),
    "smatch+io": dict(marker="o", label="smatch", c="#62A87C", ls="-", lw=2),
    "stilts+io": dict(marker="s", label="STILTS tmatch2 (TOPCAT, parallel)", c="#C0392B", ls="-", lw=2),
    "scipy+io": dict(marker="^", label="scipy cKDTree (32 threads)", c="#7F7F7F", ls=":", lw=2),
}
for key, kw in plotting_args.items():
    if key not in agg:
        continue
    d = agg[key]
    x, y = d["rows"].to_numpy(), d["median"].to_numpy()
    ax.errorbar(x, y, [y - d["q10"].to_numpy(), d["q90"].to_numpy() - y], capsize=5, **kw)

ax.set_xscale("log"); ax.set_yscale("log")
ax.tick_params(axis="both", which="major", labelsize=18, length=12, direction="in")
ax.tick_params(axis="both", which="minor", labelsize=12, length=6, direction="in")
ax.set_xlabel("Number of Crossmatched Rows")
ax.set_ylabel("Time taken (s)")
ax.set_title("Crossmatch Performance by Tool")
ax.legend(loc="upper left")

s = settings_all
v = s["versions"]
threads = {str(r["workers"]): r["threads_per_worker"] for r in recs_all if r["tool"] == "lsdb"}
mode = "results written to parquet" if s.get("output_mode") == "write" else "results counted in memory"
band = f", ZTF partitions {s['rows_band'][0]/1e6:.1f}-{s['rows_band'][1]/1e6:.1f}M rows" if s.get("rows_band") else ""
caption = (f"SLAC SDF {s['hostname']}, {s['n_cpus']} cores, {s['mem_cap_gb']:.0f} GB cap, warm page cache, {mode}{band} | "
           f"lsdb {v['lsdb']}, dask {v['dask']}, astropy {v['astropy']}, smatch {v['smatch']} | "
           f"ZTF DR23 objects x Gaia DR3, {s['radius_arcsec']:g} arcsec nearest neighbour | "
           f"{threads.get('1', 1)} thread per Dask worker | {s['timestamp_utc'][:10]}")
if settings_slurm is not None:
    caption += (f" | squares: {settings_slurm['hostname']}, {settings_slurm['n_cpus']} cores, "
                f"{settings_slurm['mem_cap_gb']:.0f} GB cap")
fig.text(0.5, 0.005, caption, ha="center", va="bottom", fontsize=9, color="0.35", wrap=True)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(OUT_PNG, dpi=300)
print("wrote", OUT_PNG)
